# Compute StreetCLIP outputs for Zero-Shot city localisation

This is done to establish feasibility of using CLIP-based models for global geolocalisation.

In [ ]:
from PIL import Image
import requests
from pathlib import Path
import os
import pickle
from glob import glob
from tqdm import tqdm
import pandas as pd
import torch

from transformers import CLIPProcessor, CLIPModel


## Load model

In [ ]:
model = CLIPModel.from_pretrained("geolocal/StreetCLIP")
processor = CLIPProcessor.from_pretrained("geolocal/StreetCLIP")
model.to("cuda")

## Setup

In [ ]:
os.makedirs("./csv_files", exist_ok=True)

In [ ]:
choices = [
    "Chicago",
    "London",
    "Los Angeles",
    "Melbourne",
    "Miami",
    "New York",
    "San Francisco",
    "Singapore",
    "Sydney",
    "Toronto",
]


**Note**: Update this path accordingly

In [ ]:
base_dir = "path_to_filtered_instacities_dataset"  # Update this path accordingly

## Running StreetCLIP

In [ ]:
for city_name in os.listdir(base_dir):
    print(f"Processing {city_name}...")
    all_image_names = []
    all_majority_cities = []

    # Load images
    all_images = glob(f"{os.path.join(base_dir, city_name)}/*.jpg")

    # Compute and store embeddings
    for image_path in tqdm(all_images):
        score_key = Path(image_path).stem
        with torch.no_grad():
            image = Image.open(image_path).convert("RGB")
            inputs = processor(
                text=choices, images=image, return_tensors="pt", padding=True
            ).to("cuda")
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)
            class_index = int(probs.argmax().item())

            all_image_names.append(score_key)
            all_majority_cities.append(choices[class_index])

    # Store scores
    df = pd.DataFrame(
        {"image_name": all_image_names, "majority_city": all_majority_cities}
    )

    df.to_csv(f"./csv_files/streetclip_{city_name}.pkl", index=False)
    torch.cuda.empty_cache()